# Avance 3 — Modelo de referencia

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Entrena las tres canalizaciones candidatas de la [1.5](../docs/decisiones.md#d1-5) sobre las matrices que dejaron el [Avance 2](../notebooks/02_Feature_Eng.ipynb) y su [compañero de actividad](../notebooks/02b_Feature_Eng_actividad.ipynb), y mide con validación cruzada agrupada por estructura.

## Qué es un modelo de referencia aquí, y qué no

**Es** el punto contra el que se comparan los modelos de los Avances 4 y 5: tres familias con sus hiperparámetros por omisión, la misma partición y las mismas métricas. **No es** un modelo afinado, ni una afirmación de desempeño final, ni un ganador — con una sola configuración por familia, decir que una gana es exactamente el error que la [1.5](../docs/decisiones.md#d1-5) ya cometió una vez y tuvo que retirar.

## Lo que este notebook hace distinto, y por qué

Las tres decisiones que todavía pueden cambiar en el proyecto son **qué se predice**, **con qué entradas** y **dónde se corta**. Ninguna vive dentro del código de aquí:

```python
t = cargar("actividad", umbral_um=1.0, representacion="descriptores+diana")
res, fdp = evaluar(canalizaciones(), t.X, t.y, t.grupos, estratos=t.estratos)
```

Cambiar el umbral es cambiar un número y cambiar la representación es cambiar una palabra. Es la lección que la fase 1 aprendió al revés: el Avance 2 congeló 171 columnas sobre una matriz ya transformada, y cambiar la transformación obliga a repetir la selección entera.

Y el escalado va **dentro** de la canalización, que se reajusta en cada pliegue. Ajustarlo antes de partir filtra la media y la varianza del pliegue de evaluación.

## 1. Preparación

In [1]:
import sys, warnings, platform
warnings.filterwarnings("ignore")
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Se inserta la raíz y no `src/`: `src.models.tareas` importa de `src.data_prep`,
# que a su vez importa de `src.features`. Es la convención del 02b y las pruebas.
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from src.evaluation.cv import intervalo_por_grupos, resumen
from src.models import cargar

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (8, 4.2)})
DATOS = RAIZ / "notebooks/datos"

print(f"Python {platform.python_version()} · pandas {pd.__version__}"
      f" · scikit-learn {sklearn.__version__}")

Python 3.12.12 · pandas 3.0.6 · scikit-learn 1.9.1


Los resultados los produce `scripts/experimento_baseline.py`, que registra cada corrida en `experimentos/` con su entorno y sus parámetros. Este notebook **no reentrena**: lee los CSV versionados, que es lo que permite reejecutarlo sin repetir horas de cómputo.

```bash
python scripts/experimento_baseline.py
python scripts/experimento_baseline.py --tarea actividad --representacion descriptores+diana
```

In [2]:
res1 = pd.read_csv(DATOS / "baseline_permeabilidad-descriptores.csv")
res2 = pd.read_csv(DATOS / "baseline_actividad-descriptores-diana-10um.csv")

for nombre, r in (("permeabilidad", res1), ("actividad", res2)):
    g = r[r.estrato.eq("todos")]
    print(f"{nombre:<14} {r.modelo.nunique()} modelos ·"
          f" {g.semilla.nunique()}×{g.pliegue.nunique()} evaluaciones ·"
          f" objetivo: {r.objetivo.iloc[0]}")

permeabilidad  3 modelos · 3×5 evaluaciones · objetivo: bbb_class (BBB+ = 1)
actividad      3 modelos · 3×5 evaluaciones · objetivo: activo a 10 µM


## 2. Fase 1 · Permeabilidad de la barrera

La matriz son los 171 descriptores que sobrevivieron al filtrado del Avance 2, **sin transformar**: la versión transformada que quedó guardada lleva Yeo-Johnson aplicado sobre todas las filas de entrenamiento, así que usarla en validación cruzada mete en cada pliegue de evaluación información ajustada con él.

La prueba reservada no se toca. Todo esto ocurre dentro de entrenamiento y validación.

In [3]:
print(resumen(res1[res1.estrato.eq("todos")]).to_string())

                      mcc         auprc_minoritaria           auroc        
                     mean     std              mean     std    mean     std
estrato modelo                                                             
todos   boosting   0.7019  0.0633            0.8843  0.0337  0.9302  0.0254
        bosque     0.6996  0.0710            0.8950  0.0356  0.9329  0.0267
        logistica  0.6866  0.0663            0.8724  0.0423  0.9133  0.0335


**La desviación de esa tabla no es un error estándar.** Las tres semillas comparten las mismas filas, así que los quince valores no son quince réplicas independientes. Para decidir si dos modelos se distinguen hace falta remuestrear **grupos**, que es la unidad independiente: los compuestos del mismo andamio comparten química y van juntos en la partición.

In [4]:
t1 = cargar("permeabilidad")
print(f"{t1}\n")

# Las predicciones fuera de pliegue se recalculan aquí solo para el intervalo;
# el ajuste ya lo hizo el script y sus cifras están en el CSV.
from src.evaluation.cv import evaluar
from src.models import canalizaciones

_, fdp1 = evaluar(canalizaciones(), t1.X, t1.y, t1.grupos, semillas=(0,), verboso=False)
filas = []
for nombre, p in fdp1.items():
    a, lo, hi = intervalo_por_grupos(t1.y, np.nanmean(p, axis=0), t1.grupos)
    filas.append({"modelo": nombre, "AUROC": a, "IC 95% bajo": lo, "IC 95% alto": hi})
pd.DataFrame(filas).set_index("modelo").round(4)

Tarea(permeabilidad, 6,306×151, objetivo=bbb_class (BBB+ = 1), 65.0% positivos, 1,048 grupos)



,AUROC,IC 95% bajo,IC 95% alto
modelo,,,
logistica,0.9138,0.8889,0.9354
bosque,0.9363,0.9122,0.9545
boosting,0.9356,0.9104,0.9545


**Los intervalos de las tres se solapan**, así que el baseline no elige ganador: establece el nivel contra el que comparar. Lo único que sí distingue es la regresión logística por debajo de las dos basadas en árboles, consistente con lo que la [1.5](../docs/decisiones.md#d1-5) ya había medido.

### 2.1 Lo que infla agrupar mal

El criterio de agrupamiento no es un detalle de implementación. Agrupar solo por andamio de Bemis-Murcko, en vez de unir andamio y esqueleto de conectividad, separa estereoisómeros del mismo compuesto entre pliegues:

In [5]:
from src.data_prep.splits import SIN_ANDAMIO

andamio_solo = t1.meta["andamio"].fillna(SIN_ANDAMIO).to_numpy()
comparacion = []
for etiqueta, g in (("andamio ∪ esqueleto (el correcto)", t1.grupos),
                    ("solo andamio", andamio_solo)):
    r, f = evaluar({"bosque": canalizaciones()["bosque"]}, t1.X, t1.y, g,
                   semillas=(0,), verboso=False)
    a, lo, hi = intervalo_por_grupos(t1.y, np.nanmean(f["bosque"], axis=0), g)
    comparacion.append({"agrupamiento": etiqueta, "grupos": len(np.unique(g)),
                        "AUROC": round(a, 4), "IC bajo": round(lo, 4),
                        "IC alto": round(hi, 4),
                        "MCC": round(r[r.estrato.eq("todos")]["mcc"].mean(), 4)})
pd.DataFrame(comparacion).set_index("agrupamiento")

,grupos,AUROC,IC bajo,IC alto,MCC
agrupamiento,,,,,
andamio ∪ esqueleto (el correcto),1048,0.9363,0.9122,0.9545,0.6989
solo andamio,2239,0.9586,0.9408,0.9725,0.7694


**Agrupar solo por andamio infla el AUROC en 2.2 puntos y el MCC en 6.2.** No es un margen despreciable: es más de lo que separa a los tres modelos entre sí. Cualquier cifra de este proyecto que no diga cómo agrupó no se puede comparar con estas.

## 3. Fase 2 · Actividad en el sistema nervioso

La unidad es el **par compuesto-diana** y el objetivo se guarda como intervalo, así que el corte de «activo» es una decisión de lectura ([2.10](../docs/decisiones.md#d2-10)). Aquí va el vigente, 10 µM, con la diana como característica: sin ella el modelo no sabe contra qué se mide, y las tasas base por diana van de 34.8% a 96.9%.

In [6]:
print(resumen(res2[res2.estrato.eq("todos")]).to_string())

                      mcc         auprc_minoritaria           auroc        
                     mean     std              mean     std    mean     std
estrato modelo                                                             
todos   boosting   0.4863  0.0180            0.7115  0.0119  0.8512  0.0102
        bosque     0.4948  0.0170            0.7240  0.0109  0.8629  0.0086
        logistica  0.3871  0.0189            0.6114  0.0091  0.7952  0.0114


### 3.1 El agregado no describe a los productos naturales

Es el resultado del [control de atajo](../docs/decisiones.md#d2-3), y obliga a que ninguna métrica de esta fase se publique sin su desglose:

In [7]:
por_estrato = (res2[res2.modelo.eq("bosque")]
               .groupby("estrato")[["n", "activos", "mcc", "auroc"]]
               .mean().round(4).sort_values("n", ascending=False))
por_estrato

,n,activos,mcc,auroc
estrato,,,,
todos,10873.8,0.7139,0.4948,0.8629
solo_chembl,10134.2,0.7484,0.4624,0.8546
solo_npass,421.8,0.1239,0.0986,0.6072
mixto,317.6,0.4019,0.2036,0.6410


La diferencia entre `solo_chembl` y `solo_npass` no es ruido de un pliegue: es el modelo separando mucho peor justo los compuestos que el proyecto existe para priorizar. Lo que todavía **no** se puede afirmar es que no sirva para ellos — a igual diana solo hay tres con datos suficientes, y el cuadro es heterogéneo ([2.3](../docs/decisiones.md#d2-3)).

## 4. Qué queda establecido, y qué no

1. **Hay nivel de referencia para las dos fases**, con la misma partición, las mismas métricas y el mismo arnés, registrado en `experimentos/`.
2. **Ningún modelo gana.** Los intervalos de las tres canalizaciones se solapan en la fase 1; con una configuración por familia, afirmar un ganador sería repetir un error que este proyecto ya retiró una vez.
3. **El agrupamiento vale más que el modelo**: 2.2 puntos de AUROC entre agrupar bien y agrupar mal, contra menos de 2 entre las tres familias.
4. **El desempeño de la fase 2 no es un número**, es uno por estrato de procedencia, y la cifra que contesta la pregunta del proyecto es la de los naturales.
5. **La prueba reservada sigue sin mirarse.** Su guarda es el [#54](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/54) y espera al Avance 5; cuando se mire, hay que declarar antes que trae 117 pares de NPASS puros, así que el intervalo sobre naturales será ancho.

Lo que **no** se decide aquí, a propósito:

| Decisión | Dónde vive |
|:--|:--|
| Hiperparámetros de cada familia | Avance 4 |
| Si se corta en 10 µM o en 1 µM | [2.3](../docs/decisiones.md#d2-3); el intervalo lo hace una columna |
| Si se clasifica o se predice potencia | [2.9](../docs/decisiones.md#d2-9), del Avance 4 |
| Qué representación conviene | [1.3](../docs/decisiones.md#d1-3) y [1.7](../docs/decisiones.md#d1-7); el arnés ya las acepta como argumento |
| El punto de operación del entregable | [C.5](../docs/decisiones.md#dc-5): no se fija, se entrega la curva |